# Практическая работа № 6 (пример выполнения)
## Observer, Factory, Singleton для «Тайны исчезнувшего дива»

**Курс:** ООП для разработки компьютерных игр
**Модуль:** 6 — Паттерны: Observer, Factory, Singleton
**Проект:** «Тайна исчезнувшего дива» — мистический детектив по мотивам цикла Виктора Дашкевича о графе Аверине
**Формат:** демонстрационный блокнот с полным разбором задания
**Цель:** показать, как добавить EventBus, фабрики и Singleton в проект

---

## 📖 Контекст

После модулей 1–5 проект имеет:

| Слой | Что есть |
|---|---|
| **Абстракции** | `GameObject` (ABC), протоколы способностей |
| **Сущности** | `Detective`, `Div`, `Location`, `Clue`, `Entity` + компоненты |
| **Системы** | Командные и обновления |
| **Состояния и поведение** | `GameState` FSM, `BehaviorStrategy`, `CommandHistory` |
| **Оркестрация** | `Game`, `GameLoop`, `WorldState` |
| **Инфраструктура** | `UI`, `InputHandler`, `Renderer` |

## 🚧 Какие проблемы остались

1. **Системы связаны напрямую.** `InvestigationSystem` дёргает UI, звук, достижения.
2. **Создание объектов зашито в `Game`.** Новые типы врагов требуют правки.
3. **Общие ресурсы не управляются.** `AudioManager` и `Config` создаются по несколько раз.

## 🎯 Что демонстрирует блокнот

| Паттерн | Проблема | Улучшение |
|---|---|---|
| **Observer** | Прямые вызовы | `EventBus` с подпиской/отпиской |
| **Factory** | Создание в коде | `EnemyFactory`, `ItemFactory` |
| **Singleton** | Дубли ресурсов | `AudioManager`, `Config` |

## Ключевая идея

Три паттерна решают три разные проблемы: Observer — связь, Factory — создание, Singleton — единственность ресурсов.

In [ ]:
import sys
print(f"Python: {sys.version}")
assert sys.version_info >= (3, 10), "Требуется Python 3.10+"

---
# Шаг 1. Теория: зачем нужны три паттерна

## 1.1. Observer: события вместо вызовов

**Проблема:** `InvestigationSystem.on_clue_found` дёргает UI, звук, достижения напрямую.

**Решение:** `EventBus` — издатели публикуют события, подписчики реагируют.

## 1.2. Factory: создание через фабрику

**Проблема:** `Game.spawn_enemy` с `if/elif`.

**Решение:** `EnemyFactory.create(kind)` — новый тип без правки `Game`.

## 1.3. Singleton: единственный экземпляр

**Проблема:** `AudioManager` и `Config` создаются по несколько раз.

**Решение:** `__new__` с `Lock` — один экземпляр, потокобезопасно.

## 1.4. Как это улучшит проект

- **Observer:** новая система подписывается, не трогая издателя.
- **Factory:** новый враг — новая запись в конфиге фабрики.
- **Singleton:** один звук, одна конфигурация.

## Для углублённого изучения

- Nystrom R. «Game Programming Patterns»: [Observer](https://gameprogrammingpatterns.com/observer.html).
- [Refactoring Guru: Observer](https://refactoring.guru/design-patterns/observer), [Factory Method](https://refactoring.guru/design-patterns/factory-method), [Singleton](https://refactoring.guru/design-patterns/singleton).

---
# Шаг 2. Реализация EventBus (Observer)

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class ClueFound:
    clue_name: str
    location: str


@dataclass(frozen=True)
class EnemyDefeated:
    enemy_type: str


class EventBus:
    """Шина событий: Observer + Pub-Sub."""

    def __init__(self):
        self._subscribers = {}

    def subscribe(self, event_type, handler):
        self._subscribers.setdefault(event_type, []).append(handler)

    def unsubscribe(self, event_type, handler):
        handlers = self._subscribers.get(event_type, [])
        if handler in handlers:
            handlers.remove(handler)

    def publish(self, event):
        for handler in list(self._subscribers.get(type(event), [])):
            try:
                handler(event)
            except Exception as exc:  # не прерываем цепочку
                print(f"⚠️ Ошибка в подписчике {handler}: {exc}")


# Демонстрация
bus = EventBus()
received = []


def on_clue(event):
    received.append(f"улика {event.clue_name}")


bus.subscribe(ClueFound, on_clue)
bus.publish(ClueFound("перо", "особняк"))
bus.publish(ClueFound("след", "пустошь"))
print(f"Подписчик получил: {received}")

bus.unsubscribe(ClueFound, on_clue)
bus.publish(ClueFound("записка", "академия"))
print(f"После отписки: {received}")

**Что мы видим:** `EventBus` принимает подписку, доставляет события, обрабатывает отписку и не падает на исключении подписчика. Издатель ничего не знает о подписчике.

---
# Шаг 3. Реализация фабрик (Factory)

In [ ]:
from dataclasses import dataclass


class Enemy:
    def __init__(self, kind, hp, speed, damage):
        self.kind = kind
        self.hp = hp
        self.speed = speed
        self.damage = damage

    def __repr__(self):
        return f"Enemy({self.kind}, hp={self.hp})"


class Item:
    def __init__(self, kind, name, description):
        self.kind = kind
        self.name = name
        self.description = description

    def __repr__(self):
        return f"Item({self.name})"


@dataclass(frozen=True)
class EnemySpec:
    kind: str
    hp: int
    speed: int
    damage: int


@dataclass(frozen=True)
class ItemSpec:
    kind: str
    name: str
    description: str


class EnemyFactory:
    def __init__(self, specs):
        self._specs = specs

    def create(self, kind):
        spec = self._specs[kind]
        return Enemy(spec.kind, spec.hp, spec.speed, spec.damage)


class ItemFactory:
    def __init__(self, specs):
        self._specs = specs

    def create(self, kind):
        spec = self._specs[kind]
        return Item(spec.kind, spec.name, spec.description)


ENEMIES = {
    "гоблин": EnemySpec("гоблин", hp=50, speed=30, damage=5),
    "волк": EnemySpec("волк", hp=80, speed=60, damage=10),
    "демон": EnemySpec("демон", hp=200, speed=40, damage=25),
}

ITEMS = {
    "лупа": ItemSpec("лупа", "Лупа", "Позволяет находить скрытые улики"),
    "фонарь": ItemSpec("фонарь", "Фонарь", "Освещает тёмные локации"),
}

enemy_factory = EnemyFactory(ENEMIES)
item_factory = ItemFactory(ITEMS)

goblin = enemy_factory.create("гоблин")
demon = enemy_factory.create("демон")
magnifier = item_factory.create("лупа")

print(goblin)
print(demon)
print(magnifier)

try:
    enemy_factory.create("дракон")
except KeyError as exc:
    print(f"❌ Неизвестный тип: {exc}")

**Что мы видим:** фабрика создаёт объекты по конфигурации. Новый тип врага — новая запись в `ENEMIES`, без правки клиента. Неизвестный тип даёт понятную ошибку.

---
# Шаг 4. Реализация Singleton

In [ ]:
import threading


class AudioManager:
    """Singleton: один звуковой движок на игру."""

    _instance = None
    _lock = threading.Lock()

    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            with cls._lock:
                if cls._instance is None:
                    cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self):
        if not hasattr(self, "_volume"):
            self._volume = 1.0
            self._sounds = []

    def set_volume(self, value):
        self._volume = value

    def play(self, name):
        self._sounds.append(name)

    @property
    def volume(self):
        return self._volume


class Config:
    """Singleton: одна конфигурация на игру."""

    _instance = None
    _lock = threading.Lock()

    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            with cls._lock:
                if cls._instance is None:
                    cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self, defaults=None):
        if not hasattr(self, "_data"):
            self._data = dict(defaults or {})

    def get(self, key, default=None):
        return self._data.get(key, default)

    def set(self, key, value):
        self._data[key] = value


audio1 = AudioManager()
audio2 = AudioManager()
print(f"AudioManager один и тот же: {audio1 is audio2}")

config1 = Config({"volume": 0.8})
config2 = Config({"volume": 0.5})
print(f"Config один и тот же: {config1 is config2}")
print(f"Значение по умолчанию не перезатёрто: {config2.get('volume')}")

config1.set("difficulty", "hard")
print(f"Настройка видна через config2: {config2.get('difficulty')}")

**Что мы видим:** `AudioManager()` и `Config()` возвращают один и тот же объект, `Lock` защищает от гонки при создании из нескольких потоков. Данные конфига не перезатираются повторной инициализацией.

---
# Шаг 5. Интеграция трёх паттернов

In [ ]:
class Game:
    """Упрощённая интеграция трёх паттернов."""

    def __init__(self):
        self.event_bus = EventBus()
        self.enemy_factory = EnemyFactory(ENEMIES)
        self.item_factory = ItemFactory(ITEMS)
        self.audio = AudioManager()
        self.config = Config({"volume": 0.8})
        self.log = []

        self.event_bus.subscribe(EnemyDefeated, self._on_enemy_defeated)
        self.event_bus.subscribe(ClueFound, self._on_clue_found)

    def _on_enemy_defeated(self, event):
        self.log.append(f"побеждён {event.enemy_type}")
        self.audio.play("victory")

    def _on_clue_found(self, event):
        self.log.append(f"найдена улика {event.clue_name}")

    def spawn_enemy(self, kind):
        return self.enemy_factory.create(kind)

    def run_scenario(self):
        self.event_bus.publish(EnemyDefeated("гоблин"))
        self.event_bus.publish(ClueFound("перо", "особняк"))
        return self.log


game = Game()
enemy = game.spawn_enemy("волк")
print(f"Создан враг: {enemy}")
print(f"Лог событий: {game.run_scenario()}")
print(f"Сыгранные звуки: {game.audio._sounds}")

**Что мы видим:** `Game` использует EventBus (Observer), фабрики (Factory) и Singleton (`AudioManager`, `Config`) вместе. Системы общаются через события, объекты создаются через фабрики, ресурсы — единственны.

---
# Шаг 6. Юнит-тесты

## 6.1. Тесты EventBus

```python
def test_subscribe_and_publish():
    bus = EventBus()
    got = []
    bus.subscribe(ClueFound, lambda e: got.append(e.clue_name))
    bus.publish(ClueFound("перо", "особняк"))
    assert got == ["перо"]


def test_unsubscribe():
    bus = EventBus()
    got = []
    handler = lambda e: got.append(1)
    bus.subscribe(ClueFound, handler)
    bus.unsubscribe(ClueFound, handler)
    bus.publish(ClueFound("x", "y"))
    assert got == []
```

## 6.2. Тесты фабрик

```python
def test_create_goblin():
    factory = EnemyFactory(ENEMIES)
    enemy = factory.create("гоблин")
    assert enemy.kind == "гоблин"
    assert enemy.hp == 50


def test_unknown_raises():
    factory = EnemyFactory(ENEMIES)
    with pytest.raises(KeyError):
        factory.create("дракон")
```

## 6.3. Тесты Singleton

```python
def test_audio_singleton():
    assert AudioManager() is AudioManager()


def test_config_persists():
    cfg = Config({"volume": 0.8})
    cfg.set("difficulty", "hard")
    assert Config().get("difficulty") == "hard"
```

Запустите `pytest -v` — все тесты должны проходить.

---
# Шаг 7. Документация

## 7.1. ADR-006

```markdown
# ADR-006: Observer, Factory, Singleton

## Контекст
- Системы связаны напрямую;
- Создание объектов зашито в Game;
- Общие ресурсы дублируются.

## Решение
- Observer: EventBus с подпиской/отпиской;
- Factory: EnemyFactory, ItemFactory по конфигурации;
- Singleton: AudioManager, Config с Lock.

## Альтернативы
1. Прямые вызовы — проще, но сильная связанность.
2. Внедрение зависимостей вместо Singleton — тестируемее.

## Последствия
- ✅ Слабая связанность систем.
- ✅ Расширяемое создание объектов.
- ✅ Единственные ресурсы.
- ⚠️ Singleton — глобальное состояние, осторожно.
```

## 7.2. UML
- Observer: `EventBus` → подписчики.
- Factory: `EnemyFactory` → `Enemy`, `ItemFactory` → `Item`.
- Singleton: `AudioManager`, `Config`.

## 7.3. PEP 8

```bash
pycodestyle core/events.py core/factories.py core/singletons.py
```

Исправьте замечания.

---
# Итоги

## Что сделали

| Паттерн | Проблема | Улучшение |
|---|---|---|
| **Observer** | Прямые вызовы | `EventBus` с подпиской/отпиской |
| **Factory** | Создание в коде | `EnemyFactory`, `ItemFactory` |
| **Singleton** | Дубли ресурсов | `AudioManager`, `Config` |

## Что изменилось в проекте

**До:** системы дёргают друг друга, объекты создаются в `Game`, ресурсы дублируются.

**После:** события через EventBus, фабрики создают объекты, Singleton управляет ресурсами.

## Как это подготовит модуль 7

- **Слоистая архитектура:** домен публикует события, presentation подписывается.
- **ECS:** фабрики будут создавать сущности с компонентами.
- **async/threading:** Singleton уже потокобезопасен, EventBus пригодится для связи потоков.

## Литература

**Базовая:**
- Nystrom R. «Game Programming Patterns»: [Observer](https://gameprogrammingpatterns.com/observer.html).
- Gamma E. et al. «Design Patterns».

**Углублённая:**
- [Refactoring Guru: Observer](https://refactoring.guru/design-patterns/observer), [Factory Method](https://refactoring.guru/design-patterns/factory-method), [Abstract Factory](https://refactoring.guru/design-patterns/abstract-factory), [Singleton](https://refactoring.guru/design-patterns/singleton).
- Документация Python: [weakref](https://docs.python.org/3/library/weakref.html), [threading](https://docs.python.org/3/library/threading.html).